# 2. Frozen Chronos cache and dependence features

Chronos-2 supplies a fixed, entity-wise native quantile grid and forecast embedding at every retained forecast origin. It is never fine-tuned. This notebook calls `build_cache_from_config`, the same function used by `simcast.cli.build_cache`; it therefore uses the same data preparation, model revision, quantile repair, finite-cell PIT construction, and cache schema as the CLI.

The cache is the boundary between frozen marginal forecasting and dependence modelling. Training access cannot read the sealed test labels.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np

NOTEBOOK_DIR = Path.cwd() / 'notebooks' if (Path.cwd() / 'notebooks').is_dir() else Path.cwd()
sys.path.insert(0, str(NOTEBOOK_DIR))
from _helpers import cache_path, resolve_config
from simcast.cli.build_cache import build_cache_from_config
from simcast.fm.cache import load_pit_library
from simcast.fm.pit import crossing_diagnostics
from simcast.fm.feature_builder import FeatureBuilder

CONFIG_FILE = 'configs/liander2024_transformer.yaml'
OVERRIDES: tuple[str, ...] = ()
BUILD_CACHE = False  # Set true only to run frozen Chronos inference and write this cache.
REBUILD_CACHE = False  # True replaces exactly the configured cache directory.
config = resolve_config(CONFIG_FILE, OVERRIDES)
CACHE_DIR = cache_path(config)
print(CACHE_DIR)

In [ ]:
if BUILD_CACHE:
    built_cache = build_cache_from_config(config, output_dir=CACHE_DIR, overwrite=REBUILD_CACHE)
    print(built_cache)
else:
    print('Cache build disabled. Set BUILD_CACHE=True after checking the resolved configuration.')

## Inspect the sealed library

The public cache contains forecasts, embeddings, split labels, and `NaN` test labels. `access='training'` is deliberately used first. The named dimensions are `[origin, entity, lead, quantile]` for quantiles and `[origin, entity, patch, hidden]` for Chronos embeddings. `lead` is one-based in the stored coordinate.

In [ ]:
library = load_pit_library(CACHE_DIR, access='training')
ds = library.dataset
display(ds)
print(library.metadata)
assert np.isnan(ds['true_y'].where(ds['split'] == 'test')).all()
display(ds[['split', 'origin_timestamp']].to_dataframe().head())

## Quantiles, crossings, and finite-cell PIT

For an observed value, the deterministic pseudo-PIT selects one of the `Q+1` cells induced by native quantiles and uses that cell's probability midpoint. It does **not** interpolate a predictive CDF. With `pit.monotone_repair: isotonic`, the same deterministic repair is applied before historical PIT construction and before scenario projection. A crossed row is invalid under `none`; an invalid entity invalidates the complete group vector for that origin and lead.

In [ ]:
predictions = ds['quantile_prediction'].values
diagnostics = crossing_diagnostics(predictions)
print('crossing rate:', diagnostics.overall)
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(ds['lead'].values, diagnostics.by_lead)
ax.set(xlabel='forecast lead', ylabel='crossing fraction', title='Native quantile crossings by lead')
ax.grid(True)

complete_valid = np.isfinite(ds['pit_z'].values).all(axis=1)
print('complete valid origin-lead vectors:', complete_valid.sum(), '/', complete_valid.size)

## Dependence feature construction

M2 and M3 receive a deterministic feature vector for each entity and lead. It combines frozen Chronos patch embeddings with configured scalar descriptors of the fixed marginal grid and location. Standardization is fit on training origins only. The checkpoint stores its fitted `FeatureBuilder` state, so evaluation does not refit it on validation or test data.

In [ ]:
features = config.features
builder = FeatureBuilder(
    output_patch_size=int(library.metadata['output_patch_size']),
    shape_eps=features.shape_eps,
    standardize_scalar_features=features.standardize_scalar_features,
    use_forecast_embedding=features.use_forecast_embedding,
    use_quantile_shape=features.use_quantile_shape,
    use_median=features.use_median,
    use_log_spread=features.use_log_spread,
    use_within_patch_position=features.use_within_patch_position,
    use_location=features.use_location,
)
# Training code performs this same construction and fits the standardizer on train origins.
print(builder.configuration())
print('embedding shape:', ds['forecast_embedding'].shape)
print('quantile grid shape:', ds['quantile_prediction'].shape)